### Table of Contents
#### 1. Project Overview
- Analytical objectives
- Business questions
---
#### 2. Analytical Layer Development
##### 2.1 Create transaction-level base table: `vw_market_base`
##### 2.2 Create city-year aggregation view: `vw_city_year_summary`
##### 2.3 Create property-type aggregation view: `vw_landuse_summary`
---
#### 3. Market Analysis
##### Analysis 1: Overall Market Trend
- transaction volume
- price growth
- year-over-year trends
##### Analysis 2: Regional Pricing Variation
- city pricing differences
- geographic concentration
##### Analysis 3: Price Segmentation
- value
- mid-market
- premium
- luxury
##### Analysis 4: Repeat Sales Analysis
- parcel transaction history
- repeat-sale anomalies
##### Analysis 5: Sale Price vs Assessed Value
- valuation gaps
- pricing inefficiencies
---
#### 4. Final Summary
- key findings
- business implications

### Goal: 
***Analyze Nashville housing transactions to identify pricing trends, market segments, regional variation, and property-value patterns using SQL.***

### I. Setup: import libraries, and connect to SQL database

In [4]:
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

In [5]:
import pandas as pd
import sqlite3

In [6]:
# import xlsx file
file_name = "housing_final_cleaned.csv"

df = pd.read_csv(file_name, index_col=0)

In [7]:
# connect to SQL database
conn = sqlite3.connect('my_projects.db')

df.to_sql("housing_final_cleaned", conn, if_exists='replace', index=False)

56373

In [8]:
# schema overview
print("The dataset's dimension:", df.shape, "\n")
print("The datatypes of columns:\n", df.dtypes)

The dataset's dimension: (56373, 22) 

The datatypes of columns:
 LandUse_raw                     str
LandUse                         str
PropertyAddress                 str
SaleDate                        str
SalePrice                     int64
LegalReference                  str
SoldAsVacant                    str
OwnerName                       str
OwnerAddress                    str
Acreage                     float64
TaxDistrict                     str
LandValue                   float64
BuildingValue               float64
TotalValue                  float64
YearBuilt                   float64
Bedrooms                    float64
FullBath                    float64
HalfBath                    float64
is_low_price_transaction      int64
is_high_price_outlier         int64
is_bed_bath_logic_issue       int64
duplicate_rank                int64
dtype: object


### II. Create analysis-ready views

#### View 1: **`vw_market_base`**
- This view creates a transactional-level analytical base table for downstream market analysis
- This view derive additional features for pricing analysis, such as:
  - `SaleYear`
  - `SaleMonth`
  - `PropertyCity`
  - `PropertyStreetAddress`
  - `TotalBaths`
  - `PropertyAgeAtSale`
- This view also create business logic flags, such as:
  - `is_market_sale`
  - `is_low_price_transaction`
  - `is_high_price_outlier`
  - `is_bed_bath_logic_issue`

In [12]:
# view 1 (vw_market_base): trasaction-level analysis base
conn.execute("DROP VIEW IF EXISTS vw_market_base")

conn.execute(
    """CREATE VIEW vw_market_base AS
           SELECT
               ParcelID,
               PropertyAddress,
               TRIM(SUBSTR(PropertyAddress, 1, INSTR(PropertyAddress, ',') - 1)) AS PropertyStreetADDRESS,
               TRIM(SUBSTR(PropertyAddress, INSTR(PropertyAddress, ',') + 1)) AS PropertyCity,
               LandUse,
               SaleDate,
               CAST(strftime('%Y', SaleDate) AS INTEGER) AS SaleYear,
               CAST(strftime('%m', SaleDate) AS INTEGER) AS SaleMonth,
               SalePrice,
               LegalReference,
               SoldAsVacant,
               Acreage,
               LandValue,
               BuildingValue,
               TotalValue,
               YearBuilt,
               Bedrooms,
               FullBath,
               HalfBath,
               FullBath + 0.5 * HalfBath AS TotalBaths,
               CASE
                   WHEN YearBuilt IS NOT NULL 
                   THEN CAST(strftime('%Y', SaleDate) AS INTEGER) - YearBuilt
                   ELSE NULL
               END AS PropertyAgeAtSale,
               CASE
                   WHEN SalePrice < 10000 THEN 0
                   WHEN is_low_price_transaction = 1 THEN 0
                   WHEN is_bed_bath_logic_issue = 1 THEN 0
                   ELSE 1
               END AS is_market_sale,
               is_low_price_transaction,
               is_high_price_outlier,
               is_bed_bath_logic_issue
           FROM housing_final""")

conn.commit()

In [13]:
# validate view 1 (vw_market_base)
pd.read_sql_query(
    """SELECT *
       FROM vw_market_base
       LIMIT 5""",
    conn)

,ParcelID,PropertyAddress,PropertyStreetADDRESS,PropertyCity,LandUse,SaleDate,SaleYear,SaleMonth,SalePrice,LegalReference,...,YearBuilt,Bedrooms,FullBath,HalfBath,TotalBaths,PropertyAgeAtSale,is_market_sale,is_low_price_transaction,is_high_price_outlier,is_bed_bath_logic_issue
0,007 00 0 125.00,"1808 FOX CHASE DR, GOODLETTSVILLE",1808 FOX CHASE DR,GOODLETTSVILLE,SINGLE FAMILY,2013-04-09,2013,4,240000,20130412-0036474,...,1986,3,3,0,3.0,27,1,0,0,0
1,007 00 0 130.00,"1832 FOX CHASE DR, GOODLETTSVILLE",1832 FOX CHASE DR,GOODLETTSVILLE,SINGLE FAMILY,2014-06-10,2014,6,366000,20140619-0053768,...,1998,3,3,2,4.0,16,1,0,0,0
2,007 00 0 138.00,"1864 FOX CHASE DR, GOODLETTSVILLE",1864 FOX CHASE DR,GOODLETTSVILLE,SINGLE FAMILY,2016-09-26,2016,9,435000,20160927-0101718,...,1987,4,3,0,3.0,29,1,0,0,0
3,007 00 0 143.00,"1853 FOX CHASE DR, GOODLETTSVILLE",1853 FOX CHASE DR,GOODLETTSVILLE,SINGLE FAMILY,2016-01-29,2016,1,255000,20160129-0008913,...,1985,3,3,0,3.0,31,1,0,0,0
4,007 00 0 149.00,"1829 FOX CHASE DR, GOODLETTSVILLE",1829 FOX CHASE DR,GOODLETTSVILLE,SINGLE FAMILY,2014-10-10,2014,10,278000,20141015-0095255,...,1984,4,3,0,3.0,30,1,0,0,0


#### View 2: **`vw_city_year_summary`**
- This view aggregates transaction-level data into city-by-year market summaries
- This view is used to analyze regional pricing differences across cities, track transactions activity over time, and compare market behavior across geographic segments
- This view contains key metrics, such as:
  - transaction volumne
  - average sale price
  - min / max sale price
  - low-price transactions
  - high-price outliers
  - bed / bath anomalies

In [15]:
conn.execute("DROP VIEW IF EXISTS vw_city_year_summary")

conn.execute(
    """CREATE VIEW vw_city_year_summary AS
           SELECT
               PropertyCity,
               SaleYear,
               COUNT(*) AS transaction_count,
               ROUND(AVG(SalePrice), 2) AS avg_sale_price,
               MIN(SalePrice) AS min_sale_price,
               MAX(SalePrice) AS max_sale_price,
               SUM(is_low_price_transaction) AS low_price_transactions,
               SUM(is_high_price_outlier) AS high_price_outliers,
               SUM(is_bed_bath_logic_issue) AS bed_bath_logic_issues
           FROM vw_market_base
           WHERE PropertyCity IS NOT NULL
           GROUP BY PropertyCity, SaleYear""")

conn.commit()

In [16]:
pd.read_sql_query(
    """SELECT *
       FROM vw_city_year_summary
       ORDER BY SaleYear, PropertyCity
       LIMIT 10""", 
    conn)

,PropertyCity,SaleYear,transaction_count,avg_sale_price,min_sale_price,max_sale_price,low_price_transactions,high_price_outliers,bed_bath_logic_issues
0,ANTIOCH,2013,1257,216937.50,3500,900000,3,78,0
1,BRENTWOOD,2013,357,292749.62,35000,2562600,0,13,0
2,FRANKLIN,2013,1,150000.00,150000,150000,0,0,0
3,GOODLETTSVILLE,2013,119,152123.88,20000,585000,0,0,0
4,HERMITAGE,2013,652,170877.81,17000,4005462,0,1,0
5,JOELTON,2013,3,122800.00,103500,145000,0,0,0
6,MADISON,2013,334,111732.90,6000,475000,2,0,0
7,MOUNT JULIET,2013,29,221658.90,64591,362880,0,0,0
8,NASHVILLE,2013,8198,261786.50,50,10750000,34,472,0
9,NOLENSVILLE,2013,56,267498.02,65000,440140,0,0,0


#### View 3: **`vw_landuse_summary`**
- This view summarizes pricing behavior across different property types
- This view is used to compare transaction volume across property categories, understand pricing differences by property type, and evaluate how anomalies are distributed across lang-use segments
- This view contains key metrics, such as:
  - transaction count
  - average sale price
  - price ranges
  - outlier rates
  - average acreage
  - average bedroom count
  - average bathroom count

In [17]:
conn.execute("DROP VIEW IF EXISTS vw_landuse_summary")

conn.execute(
    """CREATE VIEW vw_landuse_summary AS
           SELECT
               LandUse,
               COUNT(*) AS transaction_count,
               ROUND(AVG(SalePrice), 2) AS avg_sale_price,
               MIN(SalePrice) AS min_sale_price,
               MAX(SalePrice) AS max_sale_price,
               SUM(is_low_price_transaction) AS low_price_transactions,
               SUM(is_high_price_outlier) AS high_price_outliers,
               ROUND(100.0 * SUM(is_high_price_outlier) / COUNT(*), 2) AS high_price_outlier_pct,
               ROUND(AVG(CASE 
                   WHEN Acreage IS NOT NULL 
                   THEN Acreage END), 2) AS avg_acreage,
               ROUND(AVG(CASE 
                   WHEN Bedrooms IS NOT NULL 
                   THEN Bedrooms END), 2) AS avg_bedrooms,
               ROUND(AVG(CASE 
                   WHEN FullBath IS NOT NULL 
                   THEN FullBath END), 2) AS avg_full_bath
           FROM vw_market_base
           GROUP BY LandUse""")

conn.commit()

In [21]:
pd.read_sql_query(
    """SELECT *
       FROM vw_landuse_summary
       ORDER BY transaction_count DESC
       LIMIT 10""", 
    conn)

,LandUse,transaction_count,avg_sale_price,min_sale_price,max_sale_price,low_price_transactions,high_price_outliers,high_price_outlier_pct,avg_acreage,avg_bedrooms,avg_full_bath
0,SINGLE FAMILY,34120,280487.18,100,10750000,12,2102,6.16,0.46,3.06,1.86
1,RESIDENTIAL CONDO,14064,436393.02,5750,54278060,4,1060,7.54,NaN,NaN,NaN
2,VACANT RESIDENTIAL LAND,5091,333628.36,50,13156000,75,759,14.91,0.97,3.43,2.19
3,DUPLEX,1372,259955.77,12000,4400000,0,84,6.12,0.32,3.88,2.39
4,ZERO LOT LINE,1047,124392.18,12500,1210000,0,2,0.19,0.16,2.38,1.59
5,CONDO,247,1260063.77,14000,4250000,0,79,31.98,NaN,NaN,NaN
6,RESIDENTIAL COMBO/MISC,95,333821.41,3000,3625000,2,12,12.63,0.82,3.51,2.47
7,TRIPLEX,92,273588.33,8000,1333333,1,7,7.61,0.23,4.95,3.04
8,QUADPLEX,39,363625.64,50000,2665500,0,5,12.82,0.26,6.09,3.94
9,CONDOMINIUM OFC OR OTHER COM CONDO,35,1254597.14,140000,4000000,0,11,31.43,NaN,NaN,NaN


### III. Core anlayses

#### Analysis 1: Overall market trend by year

**Buiness question:**
> ##### How did Nashville housing prices change over time?

**Metrics:**
- transaction count
- average sale price
- median sale price if feasible
- Year-over-year % change

In [22]:
pd.read_sql_query(
    """WITH yearly_market AS (
           SELECT
               SaleYear,
               COUNT(*) AS transaction_count,
               ROUND(AVG(SalePrice), 2) AS avg_sale_price,
               MIN(SalePrice) AS min_sale_price,
               MAX(SalePrice) AS max_sale_price
           FROM vw_market_base
           WHERE is_market_sale = 1
           GROUP BY SaleYear)

       SELECT
           SaleYear,
           transaction_count,
           avg_sale_price,
           min_sale_price,
           max_sale_price,
           LAG(avg_sale_price) OVER (ORDER BY SaleYear) AS previous_year_avg_price,
           ROUND(
               100.0 * 
               (avg_sale_price - LAG(avg_sale_price) OVER (ORDER BY SaleYear)) /
               LAG(avg_sale_price) OVER (ORDER BY SaleYear), 2) AS yoy_price_growth_pct
       FROM yearly_market
       ORDER BY SaleYear""",
    conn)

,SaleYear,transaction_count,avg_sale_price,min_sale_price,max_sale_price,previous_year_avg_price,yoy_price_growth_pct
0,2013,11253,245406.58,10000,10750000,NaN,NaN
1,2014,14244,334868.15,10000,54278060,245406.58,36.45
2,2015,16726,400003.03,10000,14100000,334868.15,19.45
3,2016,14050,301504.66,10000,5000000,400003.03,-24.62
4,2019,2,118500.00,118500,118500,301504.66,-60.70


#### **Observations:**
- Trasaction volume increased from 11,253 (2013) to 16,726 (2015) before declining to 14,050 (2016):
  - Only 2 transactions in 2019, suggesting incomplete data and should not be interpretd
- Average sale price increased sharply:
  - increased by 36.45% in 2014
  - increased by 19.45% in 2015
- However, average sale price declined by 24.62% in 2016
  
#### **Interpretations:**
- The 2013–2015 period suggests a rapidly expanding housing market with rising transaction activity and increasing prices
- The unusually large price growth is likely partially influenced by bulk property transactions and luxury sales
- The 2016 decline appears more consistent with changes in transaction mix rather than a true market collapse
- Average price alone may overstate market movement due to sensitivity to extreme transactions
  
#### **Business takeaway:**
- Nashville experienced strong housing demand growth during 2013–2015
- Market composition materially impacts pricing metrics
- Median pricing metrics would likely provide a more stable measure for future analysis

#### Analysis 2: Regional market variation by city

**Buiness question:**
> ##### Which cities command pricing premiums relative to the broader Nashville market?

**Metrics:**
- median/average sale price by city
- transaction count
- rank by price
- price premium vs overall market

In [23]:
pd.read_sql_query(
    """WITH 
           city_summary AS (
               SELECT
                   PropertyCity,
                   COUNT(*) AS transaction_count,
                   ROUND(AVG(SalePrice), 2) AS avg_sale_price
               FROM vw_market_base
               WHERE is_market_sale = 1
               GROUP BY PropertyCity
               HAVING COUNT(*) >= 50),
               
           market_avg AS (
               SELECT ROUND(AVG(SalePrice), 2) AS overall_avg_price
               FROM vw_market_base
               WHERE is_market_sale = 1)

       SELECT
           c.PropertyCity,
           c.transaction_count,
           c.avg_sale_price,
           m.overall_avg_price,
           ROUND(c.avg_sale_price - m.overall_avg_price, 2) AS price_premium,
           RANK() OVER (
               ORDER BY c.avg_sale_price DESC) AS city_price_rank
       FROM city_summary c
       CROSS JOIN market_avg m
       ORDER BY city_price_rank""", 
    conn)

,PropertyCity,transaction_count,avg_sale_price,overall_avg_price,price_premium,city_price_rank
0,NASHVILLE,40132,367264.54,328000.9,39263.64,1
1,BRENTWOOD,1696,312258.06,328000.9,-15742.84,2
2,GOODLETTSVILLE,734,290029.91,328000.9,-37970.99,3
3,NOLENSVILLE,494,287143.72,328000.9,-40857.18,4
4,ANTIOCH,6281,252950.65,328000.9,-75050.25,5
5,MOUNT JULIET,180,243490.19,328000.9,-84510.71,6
6,HERMITAGE,3123,200090.17,328000.9,-127910.73,7
7,OLD HICKORY,1412,191477.25,328000.9,-136523.65,8
8,WHITES CREEK,97,168434.26,328000.9,-159566.64,9
9,MADISON,2111,136704.73,328000.9,-191296.17,10


#### **Observations:**
- Nashville dominated for the overwhelming majority of transactions (40,132 transactions) and had the highest average sale price (\$367K)
  - Nashville traded at approximately \$39K above the overall market average
  - Brentwood ranked second in pricing, followed by Goodlettsville and Nolensville
  - Madison, Whites Creek, Old Hickory, and Hermitage traded significantly below the overall market average
- Regional transaction volumes vary heavily across cities
  
#### **Interpretations:**
- The Nashville market serves as the dominant economic center for housing activity in this dataset
- Higher transaction volume combined with higher pricing suggests strong demand concentration in Nashville
- Lower-priced cities may represent more affordable housing markets or different property mixes
- Geographic location appears to be a major driver of housing price variation
  
#### **Business takeaway:**
- Real estate pricing strategies should be segmented geographically rather than treating Davidson County as a single market
- Nashville appears to represent the premium core market
- Lower-cost cities may offer stronger affordability opportunities for buyers or investors

#### Analysis 3: Price segmentation

**Buiness question:**
> ##### How can the market be segmented into pricing tiers?

**Output:**
- segment name
- price range
- transaction count
- dominant LandUse
- city distribution

In [24]:
pd.read_sql_query(
    """WITH price_segments AS (
           SELECT
               ParcelID,
               SalePrice,
               PropertyCity,
               LandUse,
               NTILE(4) OVER (
                   ORDER BY SalePrice) AS price_quartile
           FROM vw_market_base
           WHERE is_market_sale = 1)

       SELECT
           price_quartile,
           COUNT(*) AS transaction_count,
           MIN(SalePrice) AS min_price,
           MAX(SalePrice) AS max_price,
           ROUND(AVG(SalePrice), 2) AS avg_price,
           COUNT(DISTINCT PropertyCity) AS city_count,
           COUNT(DISTINCT LandUse) AS landuse_count
       FROM price_segments
       GROUP BY price_quartile
       ORDER BY price_quartile""",
    conn)

,price_quartile,transaction_count,min_price,max_price,avg_price,city_count,landuse_count
0,1,14069,10000,135000,91826.54,12,20
1,2,14069,135000,206000,168885.94,12,18
2,3,14069,206000,329000,259161.20,11,24
3,4,14068,329000,54278060,792162.90,10,25


#### **Observations:**
- The market was evenly segmented into four pricing tiers (~\$14K transactions each) using quartile-based segmentation
| Price Tier | Price Range | Average Price |
| :--------: | :--------: | :--------: |
| Value | \\$10K - \$135K| \$91.8K |
| Mid-market | \\$135K – \$206K | \$168.9K |
| Premium | \\$206K – \$329K | \$259.2K |
| Luxury | \\$329K – \$54.3M | \$729K |
- Higher-priced segments show:
  - slightly lower geographic coverage (`city_count`)
  - higher property type diversity (`landuse_count`)
  
#### **Interpretations:**
- The lower three segments show relatively stable pricing bands and likely represent traditional residential transactions
- The luxury segment has an extremely wide pricing range: \\$329K to \$54.3M
- This confirms what we previously identified (i.e. luxury homes, bulk condo transactions, multi-parcel development deals) are heavily concentrated in the upper segment
  - The luxury segment’s average price (\$792K) is heavily inflated by extreme transactions, making average less representative for this tier
  - Higher property-type diversity in premium/luxury segments suggests that expensive transactions are driven by a broader mix of condos, development properties, and specialized real estate assets
  
#### **Business takeaway:**
- The Nashville housing market is highly segmented rather than uniformly priced
- The lower and middle tiers likely represent the most stable owner-occupied housing market
- The luxury segment should be analyzed separately in future pricing models because extreme transactions materially distort aggregate pricing metrics

#### Analysis 4: Repeat sales / parcel history

**Buiness question:**
> ##### How do repeat property sales perform over time?

**Metrics:**
- prior sale price
- current sale price
- absolute change
- % change
- years between sales

In [25]:
pd.read_sql_query(
    """WITH repeat_sales AS (
           SELECT
               ParcelID,
               SaleDate,
               SalePrice,
               LAG(SaleDate) OVER (
                   PARTITION BY ParcelID ORDER BY SaleDate) AS previous_sale_date,
               LAG(SalePrice) OVER (
                   PARTITION BY ParcelID ORDER BY SaleDate) AS previous_sale_price
           FROM vw_market_base
           WHERE is_market_sale = 1)
           
       SELECT
           ParcelID,
           SaleDate,
           SalePrice,
           previous_sale_date,
           previous_sale_price,
           SalePrice - previous_sale_price AS price_change,
           ROUND(
               100.0 *
               (SalePrice - previous_sale_price) /
               previous_sale_price, 2) AS price_change_pct
       FROM repeat_sales
       WHERE previous_sale_price IS NOT NULL
       ORDER BY price_change_pct DESC
       LIMIT 50""",
    conn)

,ParcelID,SaleDate,SalePrice,previous_sale_date,previous_sale_price,price_change,price_change_pct
0,093 13 0B 337.00,2014-12-17,54278060,2014-06-02,141602,54136458,38231.42
1,093 13 0B 322.00,2014-12-17,54278060,2014-08-01,158000,54120060,34253.20
2,093 13 0B 447.00,2014-12-17,54278060,2014-06-02,158000,54120060,34253.20
3,093 13 0B 448.00,2014-12-17,54278060,2014-09-15,158000,54120060,34253.20
4,093 13 0B 108.00,2014-12-17,54278060,2014-09-02,180000,54098060,30054.48
5,093 13 0B 456.00,2014-12-17,54278060,2014-11-21,190000,54088060,28467.40
6,093 13 0B 134.00,2014-12-17,54278060,2014-03-28,265000,54013060,20382.29
7,109 01 0B 020.00,2015-01-30,13156000,2014-07-30,118500,13037500,11002.11
8,109 01 0B 045.00,2015-01-30,13156000,2014-07-30,124900,13031100,10433.23
9,109 01 0B 015.00,2015-01-30,13156000,2014-07-31,125325,13030675,10397.51


#### **Observations:**
- Many repeat-sale records show extremely large price increases, such as:
  - \\$158K to \$54.3M
  - \\$118K to \$13.2M
  - \\$170K to \$5.49M
- Many of these transactions occurred within relatively short time windows
- Several records involve condos and repeated parcel structures
- Large transaction clusters align with patterns previously identified in the outlier analysis
  
#### **Interpretations:**
- These transactions likely reflect: bulk condo portfolio sales, developer acquisitions, ownership restructuring, and multi-unit transfers rather than true property appreciation
- So, ranking repeat sales purely by percentage growth overemphasizes institutional transactions.
- Parcel-level transaction history requires additional filtering to isolate normal homeowner resale behavior.
  
#### **Business takeaway:**
- Repeat-sale analysis should be segmented between:
  - institutional/bulk transactions
  - traditional residential resale transactions
- Raw transaction history alone can create misleading appreciation estimates if transaction intent is ignored.

#### Analysis 5: Sales price vs. assessed value

**Buiness question:**
> ##### Which properties sell above or below assessed value?

**Metrics:**
- sale_to_assessed_ratio = SalePrice / TotalValue
- city-level average ratio
- LandUse-level ratio
- top over-assessed / under-assessed patterns

In [26]:
# Use only records where TotalValue IS NOT NULL
pd.read_sql_query(
    """WITH 
           assessed_base AS (
               SELECT
                   ParcelID,
                   PropertyCity,
                   LandUse,
                   SaleDate,
                   SalePrice,
                   TotalValue,
                   ROUND(1.0 * SalePrice / TotalValue, 2) AS sale_to_assessed_ratio
               FROM vw_market_base
               WHERE TotalValue IS NOT NULL
               AND TotalValue > 0
               AND is_market_sale = 1),
               
           city_benchmark AS (
               SELECT
                   PropertyCity,
                   COUNT(*) AS city_transaction_count,
                   ROUND(AVG(sale_to_assessed_ratio), 2) AS city_avg_ratio
               FROM assessed_base
               GROUP BY PropertyCity),

           landuse_benchmark AS (
               SELECT
                   LandUse,
                   COUNT(*) AS landuse_transaction_count,
                   ROUND(AVG(sale_to_assessed_ratio), 2) AS landuse_avg_ratio
               FROM assessed_base
               GROUP BY LandUse)

       SELECT
           a.ParcelID,
           a.PropertyCity,
           a.LandUse,
           a.SaleDate,
           a.SalePrice,
           a.TotalValue,
           a.sale_to_assessed_ratio,
           c.city_avg_ratio,
           l.landuse_avg_ratio,
           ROUND(a.sale_to_assessed_ratio - c.city_avg_ratio, 2) AS ratio_vs_city_avg,
           ROUND(a.sale_to_assessed_ratio - l.landuse_avg_ratio, 2) AS ratio_vs_landuse_avg,
           RANK() OVER (
               ORDER BY a.sale_to_assessed_ratio DESC) AS ratio_rank_high,
           RANK() OVER (
               ORDER BY a.sale_to_assessed_ratio ASC) AS ratio_rank_low
        FROM assessed_base a
        LEFT JOIN city_benchmark c 
            ON a.PropertyCity = c.PropertyCity
        LEFT JOIN landuse_benchmark l
            ON a.LandUse = l.LandUse
        ORDER BY a.sale_to_assessed_ratio DESC
        LIMIT 50""", 
    conn)

,ParcelID,PropertyCity,LandUse,SaleDate,SalePrice,TotalValue,sale_to_assessed_ratio,city_avg_ratio,landuse_avg_ratio,ratio_vs_city_avg,ratio_vs_landuse_avg,ratio_rank_high,ratio_rank_low
0,133 02 0 038.00,NASHVILLE,VACANT COMMERCIAL LAND,2014-08-08,9500000,300.0,31666.67,4.70,3686.11,31661.97,27980.56,1,25884
1,072 01 0 015.00,NASHVILLE,VACANT RESIDENTIAL LAND,2016-02-12,322543,100.0,3225.43,4.70,24.33,3220.73,3201.10,2,25883
2,072 01 0 015.00,NASHVILLE,VACANT RESIDENTIAL LAND,2015-12-07,320000,100.0,3200.00,4.70,24.33,3195.30,3175.67,3,25882
3,073 09 0 110.00,NASHVILLE,VACANT RESIDENTIAL LAND,2015-05-01,500000,200.0,2500.00,4.70,24.33,2495.30,2475.67,4,25881
4,109 01 0 032.00,NASHVILLE,MORTUARY/CEMETERY,2016-05-06,206700,200.0,1033.50,4.70,1033.50,1028.80,0.00,5,25880
5,084 15 0 118.00,NASHVILLE,VACANT RESIDENTIAL LAND,2016-08-23,1550000,1500.0,1033.33,4.70,24.33,1028.63,1009.00,6,25879
6,092 09 0 289.00,NASHVILLE,VACANT RESIDENTIAL LAND,2015-09-29,4400000,5000.0,880.00,4.70,24.33,875.30,855.67,7,25878
7,032 15 0 020.00,NASHVILLE,VACANT RESIDENTIAL LAND,2013-12-06,169900,200.0,849.50,4.70,24.33,844.80,825.17,8,25877
8,095 08 0 087.00,NASHVILLE,VACANT RESIDENTIAL LAND,2016-03-01,289900,400.0,724.75,4.70,24.33,720.05,700.42,9,25876
9,098 00 0 045.00,HERMITAGE,VACANT RESIDENTIAL LAND,2014-12-03,345000,500.0,690.00,2.11,24.33,687.89,665.67,10,25875


In [31]:
pd.read_sql_query(
    """WITH assessed_base AS (
           SELECT
               PropertyCity,
               LandUse,
               SalePrice,
               TotalValue,
               ROUND(1.0 * SalePrice / TotalValue, 2) AS sale_to_assessed_ratio
           FROM vw_market_base
           WHERE TotalValue IS NOT NULL
               AND TotalValue > 0
               AND is_market_sale = 1)

       SELECT
           COUNT(*) AS assessed_record_count,
           ROUND(AVG(sale_to_assessed_ratio), 2) AS avg_sale_to_assessed_ratio,
           MIN(sale_to_assessed_ratio) AS min_ratio,
           MAX(sale_to_assessed_ratio) AS max_ratio,
           SUM(CASE 
               WHEN sale_to_assessed_ratio > 1 
               THEN 1 
               ELSE 0 END) AS sold_above_assessed_count,
           ROUND(
               100.0 * 
               SUM(CASE 
                   WHEN sale_to_assessed_ratio > 1 
                   THEN 1 
                   ELSE 0 END) / 
               COUNT(*), 2) AS sold_above_assessed_pct,
           SUM(CASE 
               WHEN sale_to_assessed_ratio < 1 
               THEN 1 
               ELSE 0 END) AS sold_below_assessed_count,
           ROUND(
               100.0 * 
               SUM(CASE 
                   WHEN sale_to_assessed_ratio < 1 
                   THEN 1 
                   ELSE 0 END) / 
               COUNT(*), 2) AS sold_below_assessed_pct
       FROM assessed_base""", 
    conn)

,assessed_record_count,avg_sale_to_assessed_ratio,min_ratio,max_ratio,sold_above_assessed_count,sold_above_assessed_pct,sold_below_assessed_count,sold_below_assessed_pct
0,25884,4.17,0.03,31666.67,19340,74.72,6201,23.96


In [32]:
# city-level summary
pd.read_sql_query(
    """WITH assessed_base AS (
           SELECT
               PropertyCity,
               SalePrice,
               TotalValue,
               ROUND(1.0 * SalePrice / TotalValue, 2) AS sale_to_assessed_ratio
           FROM vw_market_base
           WHERE TotalValue IS NOT NULL
               AND TotalValue > 0
               AND is_market_sale = 1)

       SELECT
           PropertyCity,
           COUNT(*) AS transaction_count,
           ROUND(AVG(sale_to_assessed_ratio), 2) AS avg_sale_to_assessed_ratio,
           SUM(CASE 
               WHEN sale_to_assessed_ratio > 1 
               THEN 1 
               ELSE 0 END) AS above_assessed_count,
           ROUND(
               100.0 * 
               SUM(CASE 
                   WHEN sale_to_assessed_ratio > 1 
                   THEN 1 
                   ELSE 0 END) / 
               COUNT(*), 2) AS above_assessed_pct
       FROM assessed_base
       GROUP BY PropertyCity
       HAVING COUNT(*) >= 50
       ORDER BY avg_sale_to_assessed_ratio DESC""", 
    conn)

,PropertyCity,transaction_count,avg_sale_to_assessed_ratio,above_assessed_count,above_assessed_pct
0,NASHVILLE,20588,4.70,15422,74.91
1,ANTIOCH,1300,2.58,1065,81.92
2,MADISON,1307,2.45,849,64.96
3,BRENTWOOD,204,2.28,156,76.47
4,HERMITAGE,1052,2.11,847,80.51
5,OLD HICKORY,917,1.55,634,69.14
6,GOODLETTSVILLE,471,1.15,339,71.97


#### **Observations:**
- Across all valid transactions:
  - average sale-to-assessed ratio = 4.17x
  - ~74.7% of properties sold above assessed value
- The highest ratios were overwhelmingly concentrated in: `VACANT RESIDENTIAL LAND` and `VACANT COMMERCIAL LAND`
- Many extreme cases involved unusually low assessed values (\\$100 – \$500)
- Nashville recorded the highest average sale-to-assessed ratio (4.7x) among major cities
  
#### **Interpretations:**
- Local property assessments may lag actual market prices, particularly in fast-growing markets
- Many extreme ratios appear driven by outdated or placeholder assessor values rather than true pricing inefficiencies
- Development land transactions appear especially prone to large sale-vs-assessment gaps
- Nashville’s higher ratio may reflect stronger redevelopment activity and faster market appreciation
  
#### **Business takeaway:**
- Assessed values should not be used as a standalone proxy for market value
- Development land transactions require separate treatment when evaluating pricing efficiency
- Investors and brokers may identify redevelopment opportunities by monitoring areas where sale prices consistently exceed assessed values

**Because extreme ratios were heavily influenced by unusually low assessed values for vacant land parcels, these findings should be interpreted directionally rather than as precise valuation gaps.**

### IV. Final Summary: Key Findings & Business Insights

#### 1. This notebook was completed in two major phases:
- Phase 1: Reusable SQL views were created to support downstream analysis
  - a transaction-level analytical base table: `vw_market_base`
  - a city-level market aggregation view: `vw_city_year_summary`
  - a property-type level pricing summary: `vw_landuse_summary`

- Phase 2: The following analytical questions were explored in the core business analyses
  - Overall market trend by year
  - Regional pricing variation across cities
  - Price segmentation across value/premium tiers
  - Repeat-sale behavior and parcel transaction history
  - Sale price vs assessed value analysis


#### 2. Key findings:
- **Market growth trends:**
  - Transaction volume increased significantly from 2013 to 2015
  - Average prices rose rapidly during the same period
  - 2016 pricing softened due to transaction mix changes
  - Institutional transactions heavily influenced average pricing metrics
- **Regional variation / geographic market concentration:**
  - Nashville accounted for the majority of transactions
  - Nashville also commanded the highest pricing premium
  - Surrounding cities represented more affordable market segments
- **Price segmentation:**
  - The housing market naturally segmented into:
    - value
    - mid-market
    - premium
    - luxury tiers
  - Luxury segments showed much higher volatility due to large development transactions
- **Repeat-sale behavior:**
  - Many extreme resale gains were tied to:
    - bulk acquisitions
    - development deals
    - portfolio transfers
  - Raw appreciation metrics can be misleading without transaction filtering
- **Sale price vs. assessed value:**
  - Most transactions sold above assessed value
  - Nashville showed the largest valuation gaps
  - Vacant land assessments were often outdated or unreliable


#### 3. Final takeaway:
- This analysis shows that Nashville’s housing market cannot be treated as a uniform market.
- Pricing behavior varies significantly across:
  - geography
  - property types
  - transaction types
  - valuation patterns
- While traditional residential transactions followed relatively stable pricing patterns, institutional transactions, development deals, and outdated property assessments introduced substantial distortion in both pricing and valuation analyses.
- These findings highlight the importance of combining strong data quality controls with business context before drawing conclusions from transactional real estate data.